In [1]:
# Transformer XL extra long
# to over come limitation of standard Transformers in capturoiing long range dependencies in squences
# still have a certian limit because of fixed length concept


In [2]:
# BERT - Bidirectional Enocder Representation from Transfomers
# open source ml
# to read from both direction
# doesnt require data in any fiex order
# train on larger amount data
#

In [3]:
pip install evaluate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import tensorflow as tf
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments
from transformers import pipeline,Trainer
import evaluate

In [5]:
df=pd.read_csv('SMSSpamCollection', sep='\t', names=['labels','text'])
df


,labels,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
df['labels']=df['labels'].map({'ham':0,'spam':1})

In [7]:
df

,labels,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will ü b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [8]:
train,test=train_test_split(df,test_size=0.2, random_state=0)

In [9]:
# convert pandas to hugging face dataset format
train_dataset=Dataset.from_pandas(train)
test_dataset=Dataset.from_pandas(test)

In [10]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

In [11]:
def tokenizer_function(examples):
  return tokenizer(examples['text'], padding='max_length', truncation=True,max_length=128)

In [12]:
# apply the tokenizer to datasets
train_dataset = train_dataset.map(tokenizer_function,batched=True)
test_dataset = test_dataset.map(tokenizer_function,batched=True)

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [13]:
train_dataset=train_dataset.remove_columns(['text','__index_level_0__'])
test_dataset=test_dataset.remove_columns(['text','__index_level_0__'])

In [14]:
# load and import model
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
# define the evaluation metric
matric=evaluate.load('accuracy')

In [16]:
def compute_metric(eval_pred):
  logits,labels=eval_pred
  predictions=np.argmax(logits,axis=-1)
  return matric.compute(predictions=predictions,references=labels)

In [17]:
%pip install transformers[torch]\
%pip install 'accelerate>=1.1.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '%pip': Expected package name at the start of dependency specifier
    %pip
    ^


In [18]:
# def traning arguments
training_args = TrainingArguments(
    output_dir='./results',
    
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    eval_steps=500,
    save_strategy='epoch',
    save_steps=500
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metric
)

In [ ]:
trainer.train()

c:\Users\PGCP-AI\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
# pickle
# to seearlize or store the model in file
# store in binary file and no need to shear code just shear model

In [ ]:
# save the model
model.save_pretrained('./sms_spam_model')
tokenizer.save_pretrained('./sms_spam_tokenizer')

In [ ]:
# load model at diff location
spam_classifier=pipeline('text-classification',
                         model='./sms_spam_model',
                         tokenizer='./sms_spam_tokenizer'
                        )

In [ ]:
new_messages=[
    'Hey Rahul, are you still in the meeting? Let us meet at lunch today'
    'URGRNT! You have been selected to recive a $900 price reward! Reply earliest'
]

In [ ]:
predictions=spam_classifier(new_messages)
print(predictions)